In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_data = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv', header = None)
train_labels = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv', header = None)
test_data =  pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv', header = None)

train_labels.head()

In [ ]:
train_data.head()

In [ ]:
train_data.info()

In [ ]:
train_data.describe()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_data = scaler.fit_transform(train_data)
test_data = scaler.transform(test_data)

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(train_data, train_labels, test_size = 0.25, random_state = 1)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score 

nb_model = GaussianNB()
nb_model.fit(x_train, y_train.values.ravel())
prediction = nb_model.predict(x_test)
nb_acc = accuracy_score(y_test, prediction)
nb_acc

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier()
knn_model.fit(x_train, y_train.values.ravel())
prediction = knn_model.predict(x_test)
knn_acc = accuracy_score(y_test, prediction)
knn_acc

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=2)
rf_model.fit(x_train, y_train.values.ravel())
prediction = rf_model.predict(x_test)
rf_acc = accuracy_score(y_test, prediction)
rf_acc

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier()
dt_model.fit(x_train, y_train.values.ravel())
prediction = dt_model.predict(x_test)
dt_acc = accuracy_score(y_test, prediction)
dt_acc

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier()
xgb_model.fit(x_train, y_train.values.ravel())
prediction = xgb_model.predict(x_test)
xgb_acc = accuracy_score(y_test, prediction)
xgb_acc

In [ ]:
from sklearn.model_selection import cross_val_score

nb_cross_val = cross_val_score(nb_model, train_data, train_labels.values.ravel(), cv=10).mean()
knn_cross_val = cross_val_score(knn_model, train_data, train_labels.values.ravel(), cv=10).mean()
rf_cross_val = cross_val_score(rf_model, train_data, train_labels.values.ravel(), cv=10).mean()
dt_cross_val = cross_val_score(dt_model, train_data, train_labels.values.ravel(), cv=10).mean()
xgb_cross_val = cross_val_score(xgb_model, train_data, train_labels.values.ravel(), cv=10).mean()
print('nb model: ', nb_cross_val)
print('knn model: ', knn_cross_val)
print('rf model: ', rf_cross_val)
print('dt model: ', dt_cross_val)
print('xgb model: ', xgb_cross_val)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)
pca_train_data = pca.fit_transform(train_data)
len(pca.explained_variance_ratio_)

In [ ]:
pca_nb_cross_val = cross_val_score(nb_model, pca_train_data, train_labels.values.ravel(), cv=10).mean()
pca_knn_cross_val = cross_val_score(knn_model, pca_train_data, train_labels.values.ravel(), cv=10).mean()
pca_rf_cross_val = cross_val_score(rf_model, pca_train_data, train_labels.values.ravel(), cv=10).mean()
pca_dt_cross_val = cross_val_score(dt_model, pca_train_data, train_labels.values.ravel(), cv=10).mean()
pca_xgb_cross_val = cross_val_score(xgb_model, pca_train_data, train_labels.values.ravel(), cv=10).mean()
print('nb model: ', pca_nb_cross_val)
print('knn model: ', pca_knn_cross_val)
print('rf model: ', pca_rf_cross_val)
print('dt model: ', pca_dt_cross_val)
print('xgb model: ', pca_xgb_cross_val)

In [ ]:
from sklearn.model_selection import GridSearchCV

n_estimators = [10, 50, 100, 200, 400]
max_depth = [3, 10, 20, 40]
param_grid = dict(n_estimators=n_estimators, max_depth=max_depth)

grid_search_xgb = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=10, scoring='accuracy', n_jobs=-1)
grid_search_xgb.fit(train_data, train_labels.values.ravel())
xgb_best = grid_search_xgb.best_estimator_

print('Random Forest Best Score', grid_search_xgb.best_score_)
print('Random Forest Best Params', grid_search_xgb.best_params_)
cross_val_score(xgb_best, train_data, train_labels.values.ravel(), cv=10).mean()

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier()
xgb_model.fit(x_train, y_train.values.ravel())
prediction = xgb_model.predict(x_test)
xgb_acc = accuracy_score(y_test, prediction)
xgb_acc

In [ ]:
pred = xgb_model.predict(test_data)
rfc_best_pred = pd.DataFrame(pred)
rfc_best_pred

In [ ]:
rfc_best_pred.index += 1

rfc_best_pred.columns = ['Solution']
rfc_best_pred['Id'] = np.arange(1, rfc_best_pred.shape[0]+1)
rfc_best_pred = rfc_best_pred[['Id', 'Solution']]

rfc_best_pred.to_csv('Submission.csv',index=False)